# Detection Baseline — VisDrone2019-DET (v2, fixed class mapping)

**Course:** Computer Vision Engineering for DefenceTech  
**Assignment:** Detection Baseline Report  
**Author:** Serhii Polishchuk  

**Fixes vs v1 (mentor feedback):**
1. Explicit VisDrone → COCO class mapping (not just id −1)
2. mAP50-95 reported separately for small / medium / large objects

Pipeline:
1. Baseline YOLOv8n (COCO pretrained), imgsz=640
2. Size-based metrics (small / medium / large)
3. Controlled change: imgsz 640 → 1280
4. Before / after comparison

## 0. Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless

import os, time, json, random, copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from ultralytics import YOLO
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    HARDWARE = f"GPU: {torch.cuda.get_device_name(0)}"
else:
    HARDWARE = "CPU"
print("Hardware:", HARDWARE)

## 1. Dataset paths

In [ ]:
DATA_ROOT = Path("./datasets/VisDrone")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

URLS = {
    "VisDrone2019-DET-train": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip",
    "VisDrone2019-DET-val":   "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip",
    "VisDrone2019-DET-test-dev": "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip",
}

import zipfile, urllib.request

def download_and_extract(name, url, root):
    target_dir = root / name
    zip_path = root / f"{name}.zip"
    if target_dir.exists() and any(target_dir.rglob("*.jpg")):
        print(f"[SKIP] {name} already exists")
        return
    print(f"[DOWNLOAD] {name} ...")
    urllib.request.urlretrieve(url, zip_path)
    print(f"[EXTRACT]  {name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(root)
    zip_path.unlink(missing_ok=True)
    print(f"[DONE]    {name}")

for name, url in URLS.items():
    download_and_extract(name, url, DATA_ROOT)

print("\nDataset root:")
for p in sorted(DATA_ROOT.iterdir()):
    print(" -", p.name)

## 2. Class mapping VisDrone → COCO (mentor fix)

YOLOv8n is pretrained on **COCO**. VisDrone class IDs are different.
We map only classes that exist in both datasets.

| VisDrone (original id) | Name | COCO id | COCO name |
|------------------------|------|---------|-----------|
| 1 | pedestrian | 0 | person |
| 2 | people | 0 | person |
| 3 | bicycle | 1 | bicycle |
| 4 | car | 2 | car |
| 5 | van | 2 | car (closest) |
| 6 | truck | 7 | truck |
| 9 | bus | 5 | bus |
| 10 | motor | 3 | motorcycle |

Skipped (no COCO equivalent): ignored(0), tricycle(7), awning-tricycle(8), others(11).

In [ ]:
# Explicit VisDrone original category → COCO class id
VISDRONE_TO_COCO = {
    1: 0,   # pedestrian → person
    2: 0,   # people → person
    3: 1,   # bicycle → bicycle
    4: 2,   # car → car
    5: 2,   # van → car (approximate)
    6: 7,   # truck → truck
    9: 5,   # bus → bus
    10: 3,  # motor → motorcycle
}

# COCO names for the classes we use (for reporting)
COCO_ID_TO_NAME = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck",
}

print("VisDrone → COCO mapping:")
for vd, coco in sorted(VISDRONE_TO_COCO.items()):
    print(f"  VisDrone {vd:2d} → COCO {coco} ({COCO_ID_TO_NAME[coco]})")

## 3. Convert annotations with correct COCO class IDs

Labels are written with **COCO class indices** so they match YOLOv8n predictions.
data.yaml uses full COCO 80-class names from the model.

In [ ]:
def convert_visdrone_to_coco_yolo(split_dir: Path):
    """Convert VisDrone annotations → YOLO labels with COCO class ids."""
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"
    labels_dir = split_dir / "labels_coco"  # separate folder for mapped labels
    labels_dir.mkdir(exist_ok=True)

    # remove old cache if any
    for c in split_dir.glob("*.cache"):
        c.unlink()

    converted, skipped_boxes = 0, 0
    for ann_path in tqdm(list(ann_dir.glob("*.txt")), desc=f"Convert {split_dir.name}"):
        img_path = images_dir / (ann_path.stem + ".jpg")
        if not img_path.exists():
            continue
        with Image.open(img_path) as im:
            W, H = im.size

        lines_out = []
        with open(ann_path) as f:
            for row in f:
                parts = row.strip().split(",")
                if len(parts) < 6:
                    continue
                x, y, w, h = map(int, parts[:4])
                cat = int(parts[5])

                if cat not in VISDRONE_TO_COCO:
                    skipped_boxes += 1
                    continue
                if w <= 0 or h <= 0:
                    skipped_boxes += 1
                    continue

                cls = VISDRONE_TO_COCO[cat]  # COCO class id
                xc = (x + w / 2) / W
                yc = (y + h / 2) / H
                bw = w / W
                bh = h / H
                xc = min(max(xc, 0.0), 1.0)
                yc = min(max(yc, 0.0), 1.0)
                bw = min(max(bw, 0.0), 1.0)
                bh = min(max(bh, 0.0), 1.0)
                lines_out.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

        out_path = labels_dir / (ann_path.stem + ".txt")
        with open(out_path, "w") as f:
            f.writelines(lines_out)
        converted += 1

    print(f"  → {converted} files, skipped boxes (unmapped/invalid): {skipped_boxes}")
    return converted

for split_name in ["VisDrone2019-DET-train", "VisDrone2019-DET-val", "VisDrone2019-DET-test-dev"]:
    split_path = DATA_ROOT / split_name
    if split_path.exists():
        convert_visdrone_to_coco_yolo(split_path)

## 4. data.yaml with COCO class names

We point `labels` to `labels_coco` and use the full COCO name list from YOLOv8n so class indices match the model.

In [ ]:
# Load model once to get official COCO names
tmp_model = YOLO("yolov8n.pt")
coco_names = tmp_model.names  # dict {0: 'person', 1: 'bicycle', ...}

# Symlink / copy structure: Ultralytics expects labels/ next to images/
# We create a YOLO-ready layout with labels_coco as labels via yaml path trick:
# Actually ultralytics looks for labels/ parallel to images/. So we copy or rename.

def ensure_labels_link(split_dir: Path):
    """Make labels/ point to labels_coco content."""
    src = split_dir / "labels_coco"
    dst = split_dir / "labels"
    if dst.exists() and dst.is_symlink():
        dst.unlink()
    elif dst.exists() and dst.is_dir():
        # backup old wrong labels
        old = split_dir / "labels_old_v1"
        if not old.exists():
            dst.rename(old)
        else:
            import shutil
            shutil.rmtree(dst)
    if not dst.exists():
        dst.symlink_to("labels_coco")
    print(f"  {split_dir.name}: labels → labels_coco")

for split_name in ["VisDrone2019-DET-train", "VisDrone2019-DET-val", "VisDrone2019-DET-test-dev"]:
    sp = DATA_ROOT / split_name
    if sp.exists():
        ensure_labels_link(sp)

# Write yaml with all 80 COCO names (indices must match model)
names_block = "\n".join([f"  {i}: {name}" for i, name in coco_names.items()])
yaml_content = f"""
path: {DATA_ROOT.resolve()}
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

names:
{names_block}
"""
yaml_path = DATA_ROOT / "visdrone_coco.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip() + "\n")

print("Saved:", yaml_path)
print("nc =", len(coco_names))

## 5. Baseline configuration

In [ ]:
BASELINE_CONFIG = {
    "model": "yolov8n.pt",
    "model_source": "Ultralytics YOLOv8n (COCO pretrained)",
    "imgsz": 640,
    "conf": 0.25,
    "iou": 0.7,
    "eval_split": "val",
    "data_yaml": str(yaml_path),
    "hardware": HARDWARE,
    "seed": SEED,
    "class_mapping": "VisDrone→COCO explicit (see section 2)",
}

print("=== BASELINE CONFIGURATION ===")
for k, v in BASELINE_CONFIG.items():
    print(f"  {k}: {v}")

## 6. Run Baseline + extract metrics including size-based mAP

In [ ]:
model = YOLO(BASELINE_CONFIG["model"])

print("Running baseline validation (imgsz=640)...")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()

baseline_results = model.val(
    data=BASELINE_CONFIG["data_yaml"],
    imgsz=BASELINE_CONFIG["imgsz"],
    conf=BASELINE_CONFIG["conf"],
    iou=BASELINE_CONFIG["iou"],
    split="val",
    plots=False,
    verbose=True,
)

baseline_time = time.time() - t0
print(f"\nBaseline finished in {baseline_time:.1f} s")

if torch.cuda.is_available():
    baseline_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
else:
    baseline_memory_mb = None

In [ ]:
def extract_metrics(results):
    m = {
        "mAP50-95": float(results.box.map),
        "mAP50": float(results.box.map50),
        "precision": float(results.box.mp),
        "recall": float(results.box.mr),
    }
    # Size-based mAP (COCO-style) if available in this ultralytics version
    # results.box often has: map, map50, map75; some versions expose maps per area via stats
    rdict = getattr(results, "results_dict", None) or {}
    # Try common keys
    for key, alias in [
        ("metrics/mAP50-95(B)", "mAP50-95"),
    ]:
        if key in rdict:
            m["mAP50-95"] = float(rdict[key])

    # Ultralytics DetMetrics: check for small/medium/large
    # In many versions these are printed during val but stored as:
    box = results.box
    if hasattr(box, "maps") and box.maps is not None:
        pass  # per-class

    # Try to read from internal speed / mean results
    # Fallback: parse from stats if present
    return m

baseline_metrics = extract_metrics(baseline_results)
print("=== BASELINE METRICS (overall) ===")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")
if baseline_memory_mb is not None:
    print(f"  Peak GPU memory: {baseline_memory_mb:.0f} MB")

## 7. mAP50-95 for small / medium / large (mentor requirement)

COCO area rules (used by pycocotools / Ultralytics when available):
- **small**: area < 32²  (= 1024 px²)
- **medium**: 32² ≤ area < 96²
- **large**: area ≥ 96²

We compute a practical size-stratified evaluation by filtering GT boxes by area and re-running matching on the prediction dump.  
Simpler robust approach: use Ultralytics JSON export + area filtering, or compute AP from saved preds.

Below: count GT by size + attempt to read size AP from results; if not available, compute simplified size-wise recall/precision proxy from predictions on a sample, and full overall metrics already obtained.

**Primary method:** run `model.val` with `save_json=True` and use COCO evaluator size splits when possible. Fallback: manual GT size counts + overall mAP already fixed by class mapping.

In [ ]:
def count_gt_by_size(split_dir: Path):
    """Count GT boxes by COCO area rules (side≈sqrt(area))."""
    labels_dir = split_dir / "labels"
    images_dir = split_dir / "images"
    counts = {"small": 0, "medium": 0, "large": 0, "total": 0}
    for label_path in labels_dir.glob("*.txt"):
        img_path = images_dir / (label_path.stem + ".jpg")
        if not img_path.exists():
            continue
        with Image.open(img_path) as im:
            W, H = im.size
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                bw, bh = float(parts[3]), float(parts[4])
                area = bw * W * bh * H  # px^2
                counts["total"] += 1
                if area < 32 ** 2:
                    counts["small"] += 1
                elif area < 96 ** 2:
                    counts["medium"] += 1
                else:
                    counts["large"] += 1
    return counts

val_counts = count_gt_by_size(DATA_ROOT / "VisDrone2019-DET-val")
print("=== GT counts on val (COCO area rules) ===")
for k, v in val_counts.items():
    print(f"  {k}: {v}")

In [ ]:
# --- Size-stratified mAP via prediction + GT matching ---
# We evaluate AP50 (simpler, stable) and approximate mAP50-95 style mean over IoU thresholds
# for boxes filtered by GT size. This satisfies mentor requirement for separate size metrics.

def box_iou_xyxy(a, b):
    """IoU between two xyxy boxes."""
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    union = area_a + area_b - inter + 1e-6
    return inter / union

def load_gt(split_dir):
    """Load GT as list of (img_id, cls, xyxy, area, size_bin)."""
    labels_dir = split_dir / "labels"
    images_dir = split_dir / "images"
    gts = []
    for label_path in labels_dir.glob("*.txt"):
        img_id = label_path.stem
        img_path = images_dir / (img_id + ".jpg")
        if not img_path.exists():
            continue
        with Image.open(img_path) as im:
            W, H = im.size
        with open(label_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 5:
                    continue
                cls = int(float(p[0]))
                xc, yc, bw, bh = map(float, p[1:5])
                x1 = (xc - bw / 2) * W
                y1 = (yc - bh / 2) * H
                x2 = (xc + bw / 2) * W
                y2 = (yc + bh / 2) * H
                area = bw * W * bh * H
                if area < 32 ** 2:
                    size_bin = "small"
                elif area < 96 ** 2:
                    size_bin = "medium"
                else:
                    size_bin = "large"
                gts.append((img_id, cls, [x1, y1, x2, y2], area, size_bin))
    return gts

def predict_val(model, split_dir, imgsz, conf, iou_nms):
    """Run model on all val images, return list of (img_id, cls, conf, xyxy)."""
    images_dir = split_dir / "images"
    preds = []
    img_paths = sorted(images_dir.glob("*.jpg"))
    for img_path in tqdm(img_paths, desc=f"Predict imgsz={imgsz}"):
        r = model.predict(str(img_path), imgsz=imgsz, conf=conf, iou=iou_nms, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for b in r.boxes:
            cls = int(b.cls.item())
            cf = float(b.conf.item())
            xyxy = b.xyxy[0].cpu().numpy().tolist()
            preds.append((img_path.stem, cls, cf, xyxy))
    return preds

def ap_for_size(gts, preds, size_bin, iou_thresholds=None):
    """Compute mean AP over IoU thresholds for one size bin (class-agnostic pooling for stability)."""
    if iou_thresholds is None:
        iou_thresholds = np.arange(0.5, 1.0, 0.05)  # 50:5:95

    gt_bin = [g for g in gts if g[4] == size_bin]
    if len(gt_bin) == 0:
        return 0.0

    # index GT by image
    gt_by_img = defaultdict(list)
    for i, g in enumerate(gt_bin):
        gt_by_img[g[0]].append((i, g[1], g[2]))  # idx, cls, xyxy

    # sort preds by confidence
    preds_sorted = sorted(preds, key=lambda x: -x[2])

    aps = []
    for iou_thr in iou_thresholds:
        matched_gt = set()
        tp = []
        fp = []
        for img_id, cls, cf, xyxy in preds_sorted:
            candidates = gt_by_img.get(img_id, [])
            best_iou, best_j = 0.0, None
            for j, gcls, gbox in candidates:
                if j in matched_gt:
                    continue
                if gcls != cls:
                    continue
                iou = box_iou_xyxy(xyxy, gbox)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= iou_thr and best_j is not None:
                matched_gt.add(best_j)
                tp.append(1)
                fp.append(0)
            else:
                tp.append(0)
                fp.append(1)

        if len(tp) == 0:
            aps.append(0.0)
            continue
        tp = np.cumsum(tp)
        fp = np.cumsum(fp)
        recall = tp / max(len(gt_bin), 1)
        precision = tp / np.maximum(tp + fp, 1e-6)
        # 101-point VOC-style interpolation simplified
        mrec = np.concatenate(([0.0], recall, [1.0]))
        mpre = np.concatenate(([1.0], precision, [0.0]))
        for i in range(len(mpre) - 1, 0, -1):
            mpre[i - 1] = max(mpre[i - 1], mpre[i])
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
        aps.append(ap)

    return float(np.mean(aps))

print("Loading GT...")
gt_list = load_gt(DATA_ROOT / "VisDrone2019-DET-val")
print(f"GT boxes (mapped classes only): {len(gt_list)}")

print("Running predictions for size-stratified mAP (baseline imgsz=640)...")
preds_640 = predict_val(model, DATA_ROOT / "VisDrone2019-DET-val",
                        imgsz=640, conf=0.25, iou_nms=0.7)
print(f"Predictions: {len(preds_640)}")

size_map_baseline = {}
for sb in ["small", "medium", "large"]:
    size_map_baseline[sb] = ap_for_size(gt_list, preds_640, sb)
    print(f"  mAP50-95 proxy [{sb}]: {size_map_baseline[sb]:.4f}")

print("\n=== BASELINE SIZE mAP50-95 (approx) ===")
for k, v in size_map_baseline.items():
    print(f"  {k}: {v:.4f}")

## 8. Controlled change: imgsz 640 → 1280

In [ ]:
CHANGED_CONFIG = BASELINE_CONFIG.copy()
CHANGED_CONFIG["imgsz"] = 1280

print("=== CONTROLLED CHANGE ===")
print("  What : imgsz 640 → 1280")
print("  Why  : majority of objects are small (area < 32²); higher res gives them more pixels")
print("  Fixed: model weights, conf, iou, split, class mapping, metric code")

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model2 = YOLO(CHANGED_CONFIG["model"])

print("Running changed validation (imgsz=1280)...")
t0 = time.time()
changed_results = model2.val(
    data=CHANGED_CONFIG["data_yaml"],
    imgsz=CHANGED_CONFIG["imgsz"],
    conf=CHANGED_CONFIG["conf"],
    iou=CHANGED_CONFIG["iou"],
    split="val",
    plots=False,
    verbose=True,
)
changed_time = time.time() - t0
print(f"\nChanged finished in {changed_time:.1f} s")

changed_metrics = extract_metrics(changed_results)
if torch.cuda.is_available():
    changed_memory_mb = torch.cuda.max_memory_allocated() / 1024**2
else:
    changed_memory_mb = None

print("=== CHANGED METRICS (overall) ===")
for k, v in changed_metrics.items():
    print(f"  {k}: {v:.4f}")
if changed_memory_mb is not None:
    print(f"  Peak GPU memory: {changed_memory_mb:.0f} MB")

In [ ]:
print("Predictions for size-stratified mAP (imgsz=1280)...")
preds_1280 = predict_val(model2, DATA_ROOT / "VisDrone2019-DET-val",
                         imgsz=1280, conf=0.25, iou_nms=0.7)
print(f"Predictions: {len(preds_1280)}")

size_map_changed = {}
for sb in ["small", "medium", "large"]:
    size_map_changed[sb] = ap_for_size(gt_list, preds_1280, sb)
    print(f"  mAP50-95 proxy [{sb}]: {size_map_changed[sb]:.4f}")

print("\n=== CHANGED SIZE mAP50-95 (approx) ===")
for k, v in size_map_changed.items():
    print(f"  {k}: {v:.4f}")

## 9. Before / After tables

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "mAP50-95 (all)", "mAP50", "Precision", "Recall",
        "mAP50-95 small", "mAP50-95 medium", "mAP50-95 large",
        "Processing time (s)", "Peak GPU memory (MB)",
    ],
    "Baseline (imgsz=640)": [
        f"{baseline_metrics['mAP50-95']:.4f}",
        f"{baseline_metrics['mAP50']:.4f}",
        f"{baseline_metrics['precision']:.4f}",
        f"{baseline_metrics['recall']:.4f}",
        f"{size_map_baseline['small']:.4f}",
        f"{size_map_baseline['medium']:.4f}",
        f"{size_map_baseline['large']:.4f}",
        f"{baseline_time:.1f}",
        f"{baseline_memory_mb:.0f}" if baseline_memory_mb else "N/A",
    ],
    "Changed (imgsz=1280)": [
        f"{changed_metrics['mAP50-95']:.4f}",
        f"{changed_metrics['mAP50']:.4f}",
        f"{changed_metrics['precision']:.4f}",
        f"{changed_metrics['recall']:.4f}",
        f"{size_map_changed['small']:.4f}",
        f"{size_map_changed['medium']:.4f}",
        f"{size_map_changed['large']:.4f}",
        f"{changed_time:.1f}",
        f"{changed_memory_mb:.0f}" if changed_memory_mb else "N/A",
    ],
})

print("=== BEFORE / AFTER ===")
display(comparison)

print(f"\nΔ mAP50-95 (all): {changed_metrics['mAP50-95'] - baseline_metrics['mAP50-95']:+.4f}")
print(f"Δ mAP50-95 (small): {size_map_changed['small'] - size_map_baseline['small']:+.4f}")
if baseline_memory_mb and changed_memory_mb:
    print(f"Memory ratio: {changed_memory_mb / baseline_memory_mb:.2f}x")

## 10. Summary for the Report

In [ ]:
print("=" * 60)
print("SUMMARY FOR DETECTION BASELINE REPORT (v2)")
print("=" * 60)

print("\n1. RUN CONDITIONS")
for k, v in BASELINE_CONFIG.items():
    print(f"   {k}: {v}")

print("\n2. CLASS MAPPING (VisDrone → COCO)")
for vd, coco in sorted(VISDRONE_TO_COCO.items()):
    print(f"   VD {vd} → COCO {coco} ({COCO_ID_TO_NAME[coco]})")

print("\n3. BASELINE METRICS")
for k, v in baseline_metrics.items():
    print(f"   {k}: {v:.4f}")
print("   size mAP:", {k: round(v, 4) for k, v in size_map_baseline.items()})

print("\n4. GT COUNTS BY SIZE")
for k, v in val_counts.items():
    print(f"   {k}: {v}")

print("\n5. CONTROLLED CHANGE: imgsz 640 → 1280")
print("\n6. BEFORE / AFTER")
display(comparison)
print("\nDone.")